# 🎤 Autotune de Hatsune Miku — Corrección de pitch y formantes
### Proyecto PDDI (Procesamiento Digital de la Información) · UTEM · Prof. Jorge Vergara

**Avance 1 — Etapas 1 y 2**

Integrantes: _(completar: 2–3 personas)_

---

En este proyecto quiero tomar mi propia voz, detectar en qué nota estoy cantando y "afinarla" automáticamente, como hace el autotune. Más adelante le voy a poner el timbre de Hatsune Miku jugando con los formantes.

**Regla que me puse (y que pide el ramo):** todo se resuelve con procesamiento de señales "clásico" que vimos en clases (Fourier, DFT/FFT, STFT, convolución y filtros). **No uso nada de machine learning ni redes neuronales.**

En este primer avance dejo funcionando:
1. **Etapa 1** – la teoría del *Phase Vocoder* y cómo se conecta con lo del vault.
2. **Etapa 2** – grabar mi voz, detectar la frecuencia fundamental (f₀) y aplicar pitch shifting.

Las Etapas 3 (formantes tipo Miku) y 4 (comparaciones / espectrogramas) quedan para el próximo avance.

## Etapa 1 — La teoría: Phase Vocoder (Flanagan & Golden, 1966)

El *phase vocoder* es un algoritmo clásico (de 1966, nada de IA) para estirar o cambiar el tono de un sonido. La idea encaja con lo que vimos en la Unidad 2:

**1. Se trabaja por ventanas, no con toda la señal de una.**
En el apunte *"Espectrogramas y Señales No Estacionarias"* (Unit 2, Lec. 1) vimos que la voz es una señal **no estacionaria**: su frecuencia cambia en el tiempo, así que la FFT de toda la señal no sirve. Por eso usamos la **STFT**, que corta la señal en ventanas y le saca la FFT a cada trozo, asumiendo que dentro de cada ventana la señal es "casi estacionaria":

$$\text{STFT}[m,k]=\sum_n s[n]\,w[n-m]\,e^{-j\frac{2\pi}{N}kn}$$

**2. La fase esconde la frecuencia real (frecuencia instantánea).**
En el mismo apunte definimos la **frecuencia instantánea** como la rapidez con que cambia la fase:

$$f_i(t)=\frac{1}{2\pi}\frac{d\phi(t)}{dt}$$

El phase vocoder usa justo esto: mira cómo cambió la fase de cada bin entre una ventana y la siguiente y, con eso, calcula la frecuencia "verdadera" de cada componente (esto se llama *phase unwrapping*). Es la versión discreta de la derivada de la fase.

**3. Cambiar el tono = estirar en el tiempo + volver a muestrear.**
El truco del pitch shifting es:
- primero **estiro** la señal en el tiempo (más lenta) sin cambiar el tono, usando la STFT y arreglando las fases con la frecuencia instantánea,
- y después la **remuestreo** (`resample`) para devolverla a su duración original.

Al comprimirla de vuelta, todas las frecuencias suben, o sea **cambia el tono pero no la duración**. Si el factor es $r$, el tono queda multiplicado por $r$.

En resumen: STFT (Unit 2 L1) + derivada de la fase (frecuencia instantánea) + `resample`. Todo sale del vault.

Como calentamiento, abajo grafico el espectrograma de un **vibrato** (una nota cuyo tono oscila), que es el ejemplo del propio apunte, para *ver* cómo la frecuencia cambia en el tiempo.

In [ ]:
# Librerías: solo las de siempre 
import numpy as np
import scipy.signal
import scipy.fft
from scipy.io import wavfile
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import os

# Parámetros que voy a reusar en todo el notebook
Fs = 22050            # frecuencia de muestreo con la que trabajo (Hz)
N_VENTANA = 2048      # tamaño de ventana para la STFT (en muestras)
TRASLAPE = 0.75       # cuánto se solapan las ventanas (75%)
CARPETA_DATA = "data" # aquí guardo los .wav que voy generando

os.makedirs(CARPETA_DATA, exist_ok=True)
print("Todo listo. Fs =", Fs, "Hz")

In [ ]:
# Fabrico un vibrato: una nota de 440 Hz cuyo tono oscila (ejemplo sacado del apunte de la U2)
A_c, K, f_c, f_m = 1, 25, 440, 8          # amplitud, cuánto oscila, tono central, velocidad del vibrato
t = np.arange(0, 1.5, step=1/Fs)
vibrato = A_c*np.cos(2*np.pi*f_c*t + (K/f_m)*np.sin(2*np.pi*f_m*t))

# Le saco el espectrograma (STFT) para ver cómo la frecuencia sube y baja en el tiempo
f, tt, Sxx = scipy.signal.spectrogram(vibrato, fs=Fs, window=('kaiser', 6),
                                      nperseg=N_VENTANA, noverlap=int(N_VENTANA/1.5))
Sxx_dB = 10*np.log10(Sxx + 1e-10)         # paso a decibeles como en el apunte de voz

plt.figure(figsize=(9, 4))
plt.pcolormesh(tt, f, Sxx_dB, shading='gouraud')
plt.ylim(0, 1500)                          # me quedo con la zona baja, que es donde se ve el vibrato
plt.xlabel("Tiempo [s]"); plt.ylabel("Frecuencia [Hz]")
plt.title("Espectrograma de un vibrato: la frecuencia instantánea oscila en el tiempo")
plt.colorbar(label="Energía [dB]")
plt.show()

## Grabando mi voz en vivo

Ahora grabo mi propia voz (idealmente **una vocal sostenida**, tipo "aaaah", o una nota cantada de unos 4–6 segundos; mientras más estable el tono, mejor detecta la f₀).

El código intenta, en este orden:
1. **Grabar con el micrófono** si estoy en Google Colab,
2. si no puede, deja **subir un archivo** de audio,
3. y si estoy en Jupyter local, lee un `.wav` de la carpeta `data/`.

In [ ]:
# --- Grabación / carga de la voz -------------------------------------------------
# Nota: no sintetizo la voz, la idea es grabarla en el momento.

SEGUNDOS_GRABACION = 5
RUTA_WAV_LOCAL = os.path.join(CARPETA_DATA, "voz_grabada.wav")  # se usa si corro en Jupyter local

# ¿Estoy en Google Colab?
try:
    from google.colab import output as colab_output
    from google.colab import files as colab_files
    EN_COLAB = True
except Exception:
    EN_COLAB = False

def _convertir_a_wav(ruta_entrada, ruta_wav):
    # Colab trae ffmpeg; lo uso para dejar cualquier audio en wav mono a mi Fs
    os.system('ffmpeg -y -loglevel quiet -i "%s" -ac 1 -ar %d "%s"' % (ruta_entrada, Fs, ruta_wav))

def grabar_desde_microfono(segundos):
    # Snippet de JavaScript que enciende el micrófono y graba unos segundos
    from IPython.display import Javascript
    from base64 import b64decode
    js = Javascript('''
    var grabar = tiempo => new Promise(async resolver => {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const rec = new MediaRecorder(stream);
      const trozos = [];
      rec.ondataavailable = e => trozos.push(e.data);
      rec.start();
      await new Promise(r => setTimeout(r, tiempo));
      rec.onstop = async () => {
        const blob = new Blob(trozos);
        const lector = new FileReader();
        lector.onloadend = () => resolver(lector.result);
        lector.readAsDataURL(blob);
      };
      rec.stop();
    });
    ''')
    display(js)
    print("Grabando", segundos, "segundos... habla o canta ahora.")
    datos = colab_output.eval_js('grabar(%d)' % (segundos*1000))
    binario = b64decode(datos.split(',')[1])
    with open("grab.webm", "wb") as fbin:
        fbin.write(binario)
    _convertir_a_wav("grab.webm", RUTA_WAV_LOCAL)
    return wavfile.read(RUTA_WAV_LOCAL)

def cargar_desde_archivo_subido():
    subidos = colab_files.upload()          # abre el diálogo para elegir un audio
    nombre = list(subidos.keys())[0]
    _convertir_a_wav(nombre, RUTA_WAV_LOCAL) # lo paso a wav mono por si venía en mp3/otro
    return wavfile.read(RUTA_WAV_LOCAL)

# Intento grabar; si algo falla ofrezco subir; si no hay Colab leo un wav local
if EN_COLAB:
    try:
        fs_in, voz_cruda = grabar_desde_microfono(SEGUNDOS_GRABACION)
    except Exception as e:
        print("No pude usar el micrófono (", e, "). Súbeme un archivo de audio:")
        fs_in, voz_cruda = cargar_desde_archivo_subido()
else:
    print("No estoy en Colab: leo el wav local", RUTA_WAV_LOCAL)
    fs_in, voz_cruda = wavfile.read(RUTA_WAV_LOCAL)

print("Audio cargado:", len(voz_cruda), "muestras a", fs_in, "Hz")

In [ ]:
# Dejo el audio prolijo: mono, en float entre -1 y 1, y remuestreado a mi Fs si venía distinto
def preparar_audio(x, fs_origen, fs_destino=Fs):
    x = np.asarray(x)
    if x.ndim > 1:                       # si vino en estéreo, promedio los canales
        x = x.mean(axis=1)
    es_entero = np.issubdtype(x.dtype, np.integer)
    x = x.astype(np.float64)
    if es_entero:
        x = x / 32768.0                  # los wav enteros vienen en int16
    if fs_origen != fs_destino:          # si la frecuencia no calza, remuestreo
        x = scipy.signal.resample(x, int(round(len(x)*fs_destino/fs_origen)))
    pico = np.max(np.abs(x)) + 1e-9
    return x / pico                      # normalizo para que el volumen quede parejo

voz = preparar_audio(voz_cruda, fs_in, Fs)
wavfile.write(os.path.join(CARPETA_DATA, "voz_grabada.wav"), Fs, (voz*32767).astype(np.int16))

print("Duración:", round(len(voz)/Fs, 2), "s")
display(Audio(voz, rate=Fs))

## Etapa 2 — Detectar la frecuencia fundamental (f₀) y afinar

La **f₀** es la nota que estoy cantando (el "tono" de la voz). La detecto con la STFT: en cada ventana miro el espectro y busco el pico más bajo con energía, que corresponde al fundamental. Es el enfoque espectral que sale del vault (la autocorrelación es otra opción, pero no la vimos en clases, así que me quedo con Fourier).

Después "afino": tomo la nota promedio y la **pego a la nota más cercana** de la escala (temperamento igual, con La = 440 Hz), y uso el phase vocoder de la Etapa 1 para mover el tono hasta ahí.

In [ ]:
def detectar_f0_stft(x, fs=Fs, fmin=80, fmax=500, nperseg=N_VENTANA, umbral=0.1):
    """Recorro la señal con la STFT y en cada ventana busco la frecuencia fundamental.
    Devuelvo los tiempos y la curva de f0 (0 = ventana sin voz)."""
    noverlap = int(nperseg*TRASLAPE)
    f, t, Z = scipy.signal.stft(x, fs=fs, nperseg=nperseg, noverlap=noverlap)
    S = np.abs(Z)                                   # magnitud del espectro por ventana
    banda = (f >= fmin) & (f <= fmax)               # solo miro el rango donde vive una voz
    energia = S.sum(axis=0)
    energia_max = energia.max() + 1e-9

    f0 = np.zeros(S.shape[1])
    for j in range(S.shape[1]):
        if energia[j]/energia_max < umbral:         # ventana muy silenciosa => no hay nota
            continue
        col = S[:, j].copy()
        col[~banda] = 0
        k = int(np.argmax(col))                     # bin con más energía dentro de la banda
        if k <= 0 or k >= len(f)-1:
            continue
        # interpolación parabólica: afino el pico usando los dos bins vecinos
        a, b, c = col[k-1], col[k], col[k+1]
        denom = (a - 2*b + c)
        delta = 0.5*(a - c)/denom if denom != 0 else 0.0
        f0[j] = f[k] + delta*(f[1]-f[0])
    return t, f0

In [ ]:
tiempos_f0, f0 = detectar_f0_stft(voz)

# f0 promedio (solo de las ventanas donde sí había voz)
voz_presente = f0 > 0
f0_media = np.median(f0[voz_presente]) if voz_presente.any() else 0
print("f0 media detectada:", round(f0_media, 1), "Hz")

# Grafico la curva de f0 encima del espectrograma para chequear que detectó bien
f, tt, Sxx = scipy.signal.spectrogram(voz, fs=Fs, window=('kaiser', 6),
                                      nperseg=N_VENTANA, noverlap=int(N_VENTANA*TRASLAPE))
plt.figure(figsize=(9, 4))
plt.pcolormesh(tt, f, 10*np.log10(Sxx+1e-10), shading='gouraud')
plt.plot(tiempos_f0[voz_presente], f0[voz_presente], 'r.', label='f0 detectada')
plt.ylim(0, 1000)
plt.xlabel("Tiempo [s]"); plt.ylabel("Frecuencia [Hz]")
plt.title("Voz grabada + f0 detectada")
plt.legend(); plt.colorbar(label="Energía [dB]")
plt.show()

In [ ]:
# Paso de frecuencia (Hz) a número de nota MIDI y viceversa. La = 440 Hz = nota 69.
def hz_a_midi(fr): return 69 + 12*np.log2(fr/440.0)
def midi_a_hz(m):  return 440.0*2.0**((m-69)/12.0)

def nota_mas_cercana(fr):
    """Redondeo la frecuencia a la nota más cercana de la escala (esto es 'afinar')."""
    return midi_a_hz(round(hz_a_midi(fr)))

NOMBRES = ['Do', 'Do#', 'Re', 'Re#', 'Mi', 'Fa', 'Fa#', 'Sol', 'Sol#', 'La', 'La#', 'Si']
def nombre_nota(fr):
    m = int(round(hz_a_midi(fr)))
    return "%s%d" % (NOMBRES[m % 12], m//12 - 1)

In [ ]:
def phase_vocoder_estirar(x, rate, n_fft=N_VENTANA, hop=None):
    """Estiro la señal en el tiempo por un factor 'rate' (sin cambiar el tono todavía).
    Uso la STFT y voy corrigiendo la fase con la frecuencia instantánea (Etapa 1)."""
    if hop is None:
        hop = n_fft // 4
    win = np.hanning(n_fft)
    x = np.concatenate([np.zeros(n_fft//2), np.asarray(x, float), np.zeros(n_fft)])
    n_frames = 1 + (len(x) - n_fft)//hop

    # STFT "a mano" para tener control de las fases
    X = np.stack([np.fft.rfft(x[i*hop:i*hop+n_fft]*win) for i in range(n_frames)], axis=1)
    mag, fase = np.abs(X), np.angle(X)
    omega = 2*np.pi*np.arange(n_fft//2+1)*hop/n_fft     # avance de fase "esperado" por salto

    pasos = np.arange(0, n_frames-1, 1.0/rate)          # posiciones (fraccionarias) donde reconstruyo
    salida = np.zeros((n_fft//2+1, len(pasos)), dtype=complex)
    fase_acum = fase[:, 0].copy()
    for i, p in enumerate(pasos):
        izq = int(np.floor(p)); frac = p - izq
        der = min(izq+1, n_frames-1)
        m = (1-frac)*mag[:, izq] + frac*mag[:, der]     # interpolo la magnitud entre 2 ventanas
        dphi = fase[:, der] - fase[:, izq] - omega      # cuánto cambió la fase de más
        dphi = dphi - 2*np.pi*np.round(dphi/(2*np.pi))  # la dejo entre -pi y pi (unwrapping)
        salida[:, i] = m*np.exp(1j*fase_acum)
        fase_acum = fase_acum + omega + dphi            # acumulo con la frecuencia verdadera

    # Vuelvo al tiempo con overlap-add
    y = np.zeros(len(pasos)*hop + n_fft)
    ventana_suma = np.zeros_like(y)
    for i in range(salida.shape[1]):
        trozo = np.fft.irfft(salida[:, i], n_fft)*win
        y[i*hop:i*hop+n_fft] += trozo
        ventana_suma[i*hop:i*hop+n_fft] += win**2
    ventana_suma[ventana_suma < 1e-8] = 1e-8
    return y/ventana_suma

def cambiar_tono(x, factor):
    """Cambio el tono por 'factor' (2 = una octava arriba) sin cambiar la duración:
    estiro la señal 'factor' veces y después la comprimo de vuelta a su largo original."""
    x = np.asarray(x, float)
    n = len(x)
    if abs(factor-1.0) < 1e-3 or n < 2*N_VENTANA:
        return x
    estirada = phase_vocoder_estirar(x, factor)
    # el phase vocoder deja unas muestras de más por el relleno; recorto al largo justo (n*factor)
    largo_justo = int(round(n*factor))
    if len(estirada) >= largo_justo:
        estirada = estirada[:largo_justo]
    else:
        estirada = np.pad(estirada, (0, largo_justo - len(estirada)))
    # comprimo a n muestras: misma duración pero el tono queda multiplicado por 'factor'
    return scipy.signal.resample(estirada, n)

In [ ]:
# 1) Prueba directa del phase vocoder: subo la voz 3 semitonos para escuchar que funciona
factor_demo = 2.0**(3/12)                 # 3 semitonos hacia arriba
voz_mas_aguda = cambiar_tono(voz, factor_demo)
print("Demo: voz subida 3 semitonos")
display(Audio(voz_mas_aguda, rate=Fs))

# 2) "Autotune básico": pego el tono promedio a la nota más cercana de la escala
if f0_media > 0:
    objetivo = nota_mas_cercana(f0_media)
    factor_afinado = objetivo / f0_media
    print("Estabas en ~%.1f Hz (%s), te afino a %.1f Hz (%s), factor %.4f"
          % (f0_media, nombre_nota(f0_media), objetivo, nombre_nota(objetivo), factor_afinado))
    voz_afinada = cambiar_tono(voz, factor_afinado)
else:
    print("No detecté tono claro; dejo la voz igual.")
    voz_afinada = voz.copy()

wavfile.write(os.path.join(CARPETA_DATA, "voz_afinada.wav"), Fs,
              (np.clip(voz_afinada, -1, 1)*32767).astype(np.int16))
print("Voz afinada:")
display(Audio(voz_afinada, rate=Fs))

## Hasta aquí llega el Avance 1

Lo que quedó andando:
- **Etapa 1** – expliqué el *phase vocoder* y lo amarré a la STFT y a la frecuencia instantánea del vault, y lo dejé implementado en `phase_vocoder_estirar` / `cambiar_tono`.
- **Etapa 2** – grabo mi voz, detecto la f₀ con la STFT (`detectar_f0_stft`), la muestro sobre el espectrograma y la afino a la nota más cercana con el phase vocoder.

**Para el próximo avance:**
- **Etapa 3 – Efecto Miku:** diseñar un banco de filtros (FIR con `firwin` / IIR con `iirfilter` y `filtfilt`) en las frecuencias de los formantes para darle el timbre sintético. Ojo con algo que dice el apunte: un filtro *nunca crea frecuencias nuevas*, solo resalta o baja las que ya están, así que el banco va a acentuar formantes sobre los armónicos que ya tiene mi voz.
- **Etapa 4 – Evaluación:** comparar con Matplotlib la señal y los espectrogramas antes/después para mostrar cómo cambió el espectro.